# Espejos Coloniales — Proyecto Final
### Racismo internalizado (autoracismo) en el Perú y su raíz colonial

**Proyecto Final — Entrega 3 (versión final)**
Autor: Elvis Labaca · Curso: Prompt Engineering · Comisión N.º 96090

---

Este notebook contiene la implementación final del proyecto: un generador de piezas
educativas (texto + imagen) que conecta cinco manifestaciones actuales del autoracismo
en el Perú con su raíz colonial específica, dirigido a jóvenes peruanos de 18 a 30 años.

**Cambios respecto a la Entrega 2 (POC):**
- El **texto** se genera con la API de OpenAI (modelo texto-texto), con resultados reales
  documentados en la Sección 7.
- La **imagen** ya NO se genera por API (según lo indicado en la consigna de esta entrega
  para quien use una herramienta gratuita en lugar de DALL·E): el prompt de imagen se
  redactó con el mismo criterio que antes, pero se pegó manualmente en una herramienta
  gratuita de generación de imagen, y el resultado se adjunta en la carpeta
  [`imagenes/`](./imagenes) de este repositorio (ver Sección 8).


## 1. Instalación y configuración

In [1]:
import sys, subprocess

def _ensure(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["openai", "python-dotenv"]:
    _ensure(pkg.replace("-", "_") if pkg != "python-dotenv" else "dotenv")

print("Dependencias listas.")

Dependencias listas.


In [4]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TEXT_MODEL = "gpt-4o-mini"

API_CALLS_LOG = []

if OPENAI_API_KEY:
    print("API key detectada: se puede ejecutar en modo real (dry_run=False).")
else:
    print("No se detectó OPENAI_API_KEY. Definila en un archivo .env para generar texto real.")

API key detectada: se puede ejecutar en modo real (dry_run=False).


## 2. Técnicas de Fast Prompting utilizadas

| Técnica | Dónde se aplica | Por qué se eligió |
|---|---|---|
| **Role Prompting** | Prompt de sistema | Fija el rol de "historiador + redactor", mejora precisión histórica y tono. |
| **Few-Shot Prompting** | Ejemplo incluido en el prompt | Ancla formato y extensión (~80 palabras) entre las 5 piezas. |
| **Salida estructurada (JSON)** | Generador de texto | Une en una sola llamada lo que antes eran dos etapas (investigación + redacción). |
| **Zero-Shot Prompting** | Revisión de sensibilidad | Tarea puntual que no necesita ejemplos adicionales. |
| **Prompting negativo** | Ambos prompts (texto e imagen) | Restricciones explícitas (no mezclar raíces, evitar estereotipos) son más eficaces que solo describir lo deseado. |


## 3. Base de conocimiento del proyecto

In [5]:
TEMAS = {
    "quechua": {
        "manifestacion": "Rechazo o vergüenza al hablar quechua o aimara en espacios públicos",
        "raiz": "sistema de castas indígena",
        "contexto": "mandato colonial de hispanización como condición de ascenso social",
    },
    "alisado": {
        "manifestacion": 'Alisado capilar como condición de "buena presencia" laboral',
        "raiz": "régimen esclavista",
        "contexto": "normas de moralidad pública sobre el cabello afro impuestas desde el siglo XVIII",
    },
    "mejorar_raza": {
        "manifestacion": 'La expresión "mejorar la raza" al elegir pareja',
        "raiz": "ambos sistemas coloniales",
        "contexto": "lógica de blanqueamiento común al sistema de castas y al régimen esclavista",
    },
    "apellido": {
        "manifestacion": "Vergüenza del apellido, el acento o el origen andino/amazónico",
        "raiz": "sistema de castas indígena",
        "contexto": "jerarquización colonial por origen étnico y geográfico",
    },
    "medios": {
        "manifestacion": "Sobrerrepresentación de rasgos europeos como sinónimo de belleza en medios",
        "raiz": "ambos sistemas coloniales",
        "contexto": "convergencia histórica en el canon estético peruano actual",
    },
}

list(TEMAS.keys())

['quechua', 'alisado', 'mejorar_raza', 'apellido', 'medios']

## 4. Generador de texto (modelo texto-texto)

Una llamada a la API por pieza. Combina Role Prompting + Few-Shot + salida JSON +
restricciones explícitas.

In [6]:
SYSTEM_PROMPT = """Eres historiador especializado en el Virreinato del Perú y en la historia de la \
esclavización de personas afrodescendientes en América, y además redactor de contenido educativo \
para redes sociales. Tu tarea es producir piezas breves, rigurosas y no acusatorias sobre racismo \
internalizado en el Perú actual, conectando cada manifestación con su raíz colonial específica. \
Nunca mezcles la raíz del sistema de castas indígena con la del régimen esclavista salvo que ambas \
apliquen explícitamente. Respondes siempre en JSON válido, sin texto adicional fuera del JSON."""

FEW_SHOT_EXAMPLE_USER = """Manifestación: Vergüenza del apellido o el origen andino en contextos \
urbanos y laborales.
Raíz histórica: sistema de castas indígena.
Contexto: jerarquización colonial por origen étnico y geográfico."""

FEW_SHOT_EXAMPLE_ASSISTANT = json.dumps({
    "texto": (
        "Cambiar de apellido o esconder el pueblo de origen para 'encajar' en la ciudad no es "
        "casualidad. Desde la Colonia, ser identificado como indígena o de provincia cerraba "
        "puertas: el sistema de castas premiaba el apellido español y castigaba el origen andino. "
        "¿Alguna vez sentiste que tu apellido o tu acento decían más de vos de lo que quisiste?"
    ),
    "raiz_colonial": "sistema de castas indígena",
    "prompt_imagen": (
        "Ilustracion digital conceptual, estilo editorial, paleta tierra y ocres con acentos "
        "de azul andino. Un documento de identidad estilizado cuyo nombre se desdibuja en el aire "
        "como humo, sin rostros ni texto legible, composicion centrada, formato cuadrado. Evitar "
        "estereotipos caricaturescos de rasgos indigenas."
    ),
}, ensure_ascii=False)


def build_text_prompt(tema_key: str) -> list:
    t = TEMAS[tema_key]
    user_prompt = f"""Manifestación: {t['manifestacion']}.
Raíz histórica: {t['raiz']}.
Contexto: {t['contexto']}.

Genera una pieza para un post de Instagram educativo sobre racismo internalizado en el Perú,
dirigida a personas de 18 a 30 años sin formación académica especializada.

Reglas obligatorias:
- Estructura en 3 partes dentro de "texto": (1) situación cotidiana en tono cercano, no acusatorio;
  (2) origen colonial citando el mecanismo histórico concreto indicado arriba, sin mezclarlo con
  la otra raíz colonial salvo que el campo "Raíz histórica" diga "ambos sistemas coloniales";
  (3) una pregunta de cierre reflexiva, sin moralizar.
- Máximo 80 palabras en "texto".
- "prompt_imagen": una metáfora visual (no una persona real), estilo editorial, sin texto dentro
  de la imagen, sin estereotipos caricaturescos de rasgos indígenas o afrodescendientes.

Responde solo con este JSON, sin explicación adicional:
{{"texto": "...", "raiz_colonial": "...", "prompt_imagen": "..."}}"""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": FEW_SHOT_EXAMPLE_USER},
        {"role": "assistant", "content": FEW_SHOT_EXAMPLE_ASSISTANT},
        {"role": "user", "content": user_prompt},
    ]

In [7]:
def generate_piece(tema_key: str, dry_run: bool = True) -> dict:
    """Genera una pieza (texto + prompt de imagen) para el tema indicado.

    dry_run=True  -> no llama a la API, devuelve una salida simulada (sin costo).
    dry_run=False -> hace UNA llamada real a la API de OpenAI (Chat Completions, JSON mode).
    """
    messages = build_text_prompt(tema_key)

    if dry_run:
        t = TEMAS[tema_key]
        return {
            "texto": f"[DEMO] Pieza simulada sobre: {t['manifestacion']}. (raíz: {t['raiz']}).",
            "raiz_colonial": t["raiz"],
            "prompt_imagen": f"[DEMO] Prompt de imagen simulado para: {t['manifestacion']}.",
        }

    if not OPENAI_API_KEY:
        raise RuntimeError("Falta OPENAI_API_KEY. Define la variable de entorno o usa dry_run=True.")

    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    response = client.chat.completions.create(
        model=TEXT_MODEL,
        messages=messages,
        response_format={"type": "json_object"},
        temperature=0.7,
    )
    API_CALLS_LOG.append({"tipo": "texto", "tema": tema_key, "modelo": TEXT_MODEL})

    return json.loads(response.choices[0].message.content)

## 5. Generación de imagen — flujo manual (sin API)

Siguiendo la indicación de esta entrega, **no se usa API para la imagen**. En su lugar:

1. Se toma el `prompt_imagen` devuelto por `generate_piece()` para cada tema.
2. Se pega manualmente en una herramienta gratuita de generación de imagen (Nightcafe u otra).
3. Se descarga la imagen resultante y se guarda en la carpeta [`imagenes/`](./imagenes)
   de este repositorio, con el mismo nombre que la clave del tema
   (por ejemplo `imagenes/apellido.png`).
4. Los 5 prompts utilizados quedan documentados también en el
   [`README.md`](./README.md), junto a la imagen correspondiente, para que se pueda
   verificar que la imagen corresponde al prompt.

Esta celda solo imprime los 5 prompts de imagen listos para copiar y pegar en la
herramienta elegida (requiere haber generado antes el texto de cada pieza, real o demo,
en la Sección 7).

In [8]:
def imprimir_prompts_de_imagen(resultados: dict):
    """Recibe un diccionario {tema_key: resultado_de_generate_piece} y muestra
    los 5 prompts de imagen listos para copiar en la herramienta de generación elegida."""
    for tema_key, resultado in resultados.items():
        print(f"--- {tema_key} ---")
        print(resultado["prompt_imagen"])
        print()

## 6. Función de revisión (Zero-Shot) — opcional, bajo demanda

In [9]:
def review_piece(texto: str, dry_run: bool = True) -> dict:
    if dry_run:
        return {"observaciones": "[DEMO] Sin llamada real. Revisión disponible con dry_run=False."}

    if not OPENAI_API_KEY:
        raise RuntimeError("Falta OPENAI_API_KEY. Define la variable de entorno o usa dry_run=True.")

    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    prompt = f"""Revisa el siguiente texto de una campaña educativa sobre racismo internalizado
en el Perú. Señala en JSON si hay: (1) afirmaciones históricas no verificables,
(2) tono acusatorio hacia el lector, (3) estereotipos, (4) lenguaje poco accesible.
Responde solo: {{"observaciones": "..."}}.

Texto: \"\"\"{texto}\"\"\" """

    response = client.chat.completions.create(
        model=TEXT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0.3,
    )
    API_CALLS_LOG.append({"tipo": "revision", "modelo": TEXT_MODEL})
    return json.loads(response.choices[0].message.content)

## 7. Ejecución real — generación de las 5 piezas de texto

**Esta es la corrida que produce los resultados reales del proyecto.**
Requiere una `OPENAI_API_KEY` válida en el archivo `.env`. El costo aproximado de
generar las 5 piezas con `gpt-4o-mini` es de centavos de dólar (5 llamadas en total).



In [10]:
DRY_RUN = False  

resultados = {}
for tema_key in TEMAS:
    resultados[tema_key] = generate_piece(tema_key, dry_run=DRY_RUN)

for tema_key, r in resultados.items():
    print(f"=== {tema_key} ===")
    print("Texto:", r["texto"])
    print("Raíz colonial:", r["raiz_colonial"])
    print("Prompt de imagen:", r["prompt_imagen"])
    print()

print(f"Llamadas reales a la API en esta corrida: {len(API_CALLS_LOG)}")


=== quechua ===
Texto: ¿Te has sentido incómodo al hablar quechua o aimara en público? Esta vergüenza tiene raíces en la Colonia, donde la hispanización era el camino para ascender socialmente, despreciando nuestras lenguas originarias. ¿Cuántas veces te has callado por temor al qué dirán?
Raíz colonial: sistema de castas indígena
Prompt de imagen: Una mariposa con alas de papel que representa un libro abierto, volando sobre un fondo abstracto en tonos tierra y celeste. La mariposa simboliza la libertad y la transformación, mientras el libro sugiere el conocimiento y las raíces culturales.

=== alisado ===
Texto: ¿Cuántas veces has sentido que un peinado 'correcto' es la clave para ser tomado en serio en el trabajo? Desde el siglo XVIII, el cabello afro fue visto como poco profesional, un remanente del régimen esclavista que definía la 'buena presencia'. ¿Hasta qué punto seguimos cargando con esos prejuicios en nuestras decisiones diarias?
Raíz colonial: régimen esclavista
Prompt de im

## 8. Prompts de imagen listos para uso manual

Ejecutar después de la Sección 7 (con `resultados` ya generado) para obtener los 5
prompts de imagen listos para copiar en Nightcafe u otra herramienta gratuita.

In [ ]:
imprimir_prompts_de_imagen(resultados)

--- quechua ---
Una mariposa con alas de papel que representa un libro abierto, volando sobre un fondo abstracto en tonos tierra y celeste. La mariposa simboliza la libertad y la transformación, mientras el libro sugiere el conocimiento y las raíces culturales.

--- alisado ---
Una melena de cabello rizado que se transforma en un mar de olas, simbolizando la libertad y la identidad, en tonos de azul y negro, estilo editorial, composición centrada, sin figuras humanas.

--- mejorar_raza ---
Un laberinto de caminos entrelazados, representando decisiones y elecciones. Al fondo, un sol que se asoma, simbolizando la búsqueda de claridad. Paleta de colores cálidos y suaves, sin figuras humanas, en estilo editorial.

--- apellido ---
Un puente estilizado que conecta dos orillas, una verde y otra gris, con hojas que caen, simbolizando la conexión entre raíces y el miedo a cruzar. Estilo editorial, sin personas, enfoque en la metáfora del viaje.

--- medios ---
Una balanza antigua donde un lado

## 9. Resultados

*(Completar esta sección después de ejecutar la Sección 7 en modo real y de generar
las 5 imágenes de forma manual. Ver la plantilla y el detalle completo en el
[`README.md`](./README.md), sección "Resultados".)*

Resumen de lo que se evaluó sobre los resultados obtenidos:
- Si el texto de cada pieza respeta el límite de 80 palabras y la estructura de 3 partes pedida.
- Si la raíz colonial indicada coincide con la documentada en la base de conocimiento (Sección 3).
- Si la imagen generada manualmente corresponde razonablemente al prompt (metáfora visual,
  sin estereotipos, sin texto dentro de la imagen).
- Número real de llamadas a la API utilizadas (variable `API_CALLS_LOG`).


## 10. Conclusiones

Ver el detalle completo en el `README.md`. En síntesis, el proyecto permitió reducir a
la mitad las consultas a la API por pieza respecto al enfoque manual de la Preentrega 1,
mantener la separación entre las dos raíces coloniales trabajadas (sistema de castas
indígena y régimen esclavista) mediante restricciones explícitas en el prompt, y separar
el desarrollo del gasto real gracias al modo demo, dejando el costo acotado a una única
corrida final controlada.